# 02. Chuẩn bị dữ liệu và cố định bài toán

Notebook này tạo ra cặp `(X, y)` cố định dùng cho mọi thí nghiệm về sau, chọn
`lambda` bằng cross-validation, và tính các hằng số `L`, `mu`, `kappa`, `f*`.

Sau khi chạy xong, các file trong `data/processed/` không được sửa nữa. Mọi so
sánh giữa các thuật toán chỉ có ý nghĩa khi chúng cùng làm việc trên một hàm mục
tiêu.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.plotting import set_style

set_style()


In [ ]:
from src.data import build_design_matrix, save_processed, select_lambda

CSV_PATH = ROOT / "data" / "raw" / "used_phone_price_prediction_1M.csv"
SEED = 0
SAMPLE_ROWS = 200_000   # lấy mẫu để lưới tham số chạy được trong thời gian hợp lý

df = pd.read_csv(CSV_PATH)
design = build_design_matrix(
    df,
    target=None,             # None: tự đoán từ tên cột
    rare_threshold=10,       # gộp các mức xuất hiện dưới 10 lần
    add_interactions=True,   # tích từng cặp cột định lượng, tạo trên cột đã chuẩn hóa
    interaction_base=None,   # None: dùng toàn bộ cột định lượng
    log_target=True,
    test_size=0.2,
    seed=SEED,
    sample_rows=SAMPLE_ROWS,
)

print("cột mục tiêu:", design.target, "| log transform:", design.log_target)
print(f"train: n = {design.X_train.shape[0]}, d = {design.X_train.shape[1]}")
print(f"test : n = {design.X_test.shape[0]}")

## Chọn hệ số hiệu chỉnh

Đây là bước chọn mô hình, không phải bước tối ưu hóa. Giá trị `lambda` tìm được
ở đây sẽ được cố định cho toàn bộ phần sau.

In [ ]:
lam_grid = np.logspace(-6, 2, 25)
lam, cv_records = select_lambda(design.X_train, design.y_train, grid=lam_grid, seed=SEED, rule="one_se")
cv = pd.DataFrame(cv_records)
print(f"lambda được chọn: {lam:.6g}")

fig, ax = plt.subplots()
ax.semilogx(cv["lam"], cv["cv_mse"], marker="o", markersize=3, color="#1f77b4")
best = cv.loc[cv["cv_mse"].idxmin()]
ax.fill_between(cv["lam"], best["cv_mse"] - best["cv_mse_se"], best["cv_mse"] + best["cv_mse_se"],
                color="#cccccc", alpha=0.5, label="one standard error band")
ax.axvline(lam, color="#d62728", linestyle="--", linewidth=1.0, label=f"selected lambda = {lam:.3g}")
ax.axvline(best["lam"], color="#7f7f7f", linestyle=":", linewidth=1.0, label=f"CV minimum = {best['lam']:.3g}")
ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("5-fold CV mean squared error")
ax.set_title("Model selection for the regularization strength")
ax.legend()
fig.tight_layout()

## Cố định bài toán và tính các hằng số

In [ ]:
config = save_processed(design, lam)
for key in ("n", "d", "lam", "L", "mu", "kappa", "f_star", "w_star_norm"):
    print(f"{key:>12s} = {config[key]:.6g}")

## Ảnh hưởng của chuẩn hóa lên số điều kiện

Thí nghiệm ở mục 5.7 của kế hoạch. So sánh `kappa` của ma trận thiết kế trước và
sau khi chuẩn hóa cột, rồi chạy gradient descent trên cả hai để thấy hệ quả.

In [ ]:
from src.first_order import gradient_descent
from src.plotting import plot_comparison
from src.problem import RidgeProblem

# Unstandardized version of the same columns, centered only.
X_raw = design.X_train * 1.0
rng = np.random.default_rng(SEED)
scales = np.exp(rng.uniform(-3, 3, size=X_raw.shape[1]))
X_unscaled = X_raw * scales          # reintroduce heterogeneous column scales
X_unscaled -= X_unscaled.mean(axis=0)

problem_scaled = RidgeProblem(design.X_train, design.y_train, lam)
problem_unscaled = RidgeProblem(X_unscaled, design.y_train, lam)

print(f"kappa sau chuẩn hóa   : {problem_scaled.kappa:.4g}")
print(f"kappa chưa chuẩn hóa  : {problem_unscaled.kappa:.4g}")

In [ ]:
runs = []
labels = []
for prob, name in ((problem_scaled, "standardized"), (problem_unscaled, "unstandardized")):
    res = gradient_descent(prob, max_iter=500, record_every=5, step_rule={"kind": "fixed", "multiple": 1.0})
    # Suboptimality is measured against each problem's own optimum.
    res.params["label"] = f"GD, {name} (kappa = {prob.kappa:.3g})"
    runs.append((prob, res))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for xaxis, ax in zip(("iter", "time"), axes):
    for prob, res in runs:
        x = res.iter_hist if xaxis == "iter" else res.time_hist
        ax.semilogy(x, res.suboptimality(prob.f_star), label=res.params["label"])
    ax.set_xlabel("Iteration" if xaxis == "iter" else "Wall-clock time (s)")
    ax.set_ylabel(r"$f(w_k) - f^*$")
    ax.legend()
fig.suptitle("Effect of column standardization on the convergence rate")
fig.tight_layout()

from src.plotting import save_figure
save_figure(fig, "normalization_iter")